In [ ]:
# ============================================
# PIXELCNN FROM SCRATCH
# Fashion-MNIST
# Google Colab Version
# ============================================

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import (
    datasets,
    layers,
    models,
    optimizers,
    callbacks
)

print("TensorFlow version:", tf.__version__)

print(
    "GPU:",
    tf.config.list_physical_devices("GPU")
)

In [ ]:
# ============================================
# PARAMETERS
# ============================================

IMAGE_SIZE = 16
PIXEL_LEVELS = 4
N_FILTERS = 128
RESIDUAL_BLOCKS = 5
BATCH_SIZE = 128

# Start with 10 epochs for testing
EPOCHS = 10

# Original notebook:
# EPOCHS = 150

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Image size:", IMAGE_SIZE)
print("Pixel levels:", PIXEL_LEVELS)
print("Filters:", N_FILTERS)
print("Residual blocks:", RESIDUAL_BLOCKS)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)

In [ ]:
# ============================================
# CHECK GPU
# ============================================

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print("GPU is available!")
    print(gpus)
else:
    print("GPU is NOT available.")
    print("Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# ============================================
# LOAD FASHION-MNIST
# ============================================

(x_train, _), (x_test, _) = datasets.fashion_mnist.load_data()

print("Training images:", x_train.shape)
print("Testing images :", x_test.shape)

In [ ]:
# ============================================
# DISPLAY ORIGINAL IMAGES
# ============================================

plt.figure(figsize=(10, 5))

for i in range(10):

    plt.subplot(2, 5, i + 1)

    plt.imshow(
        x_train[i],
        cmap="gray"
    )

    plt.axis("off")

plt.suptitle("Original Fashion-MNIST Images")

plt.show()

In [ ]:
# ============================================
# PREPROCESS DATA
# ============================================

def preprocess(imgs_int):

    # Add channel dimension
    imgs_int = np.expand_dims(
        imgs_int,
        -1
    )

    # Resize 28x28 -> 16x16
    imgs_int = tf.image.resize(
        imgs_int,
        (IMAGE_SIZE, IMAGE_SIZE)
    ).numpy()

    # Convert to 4 pixel levels
    imgs_int = (
        imgs_int /
        (256 / PIXEL_LEVELS)
    ).astype(int)

    # Float version for model input
    imgs = imgs_int.astype("float32")

    imgs = imgs / PIXEL_LEVELS

    return imgs, imgs_int

In [ ]:
# ============================================
# APPLY PREPROCESSING
# ============================================

input_data, output_data = preprocess(
    x_train
)

print("Input data shape :", input_data.shape)
print("Output data shape:", output_data.shape)

print(
    "Minimum pixel value:",
    input_data.min()
)

print(
    "Maximum pixel value:",
    input_data.max()
)

In [ ]:
# ============================================
# DISPLAY PREPROCESSED IMAGES
# ============================================

plt.figure(figsize=(10, 5))

for i in range(10):

    plt.subplot(2, 5, i + 1)

    plt.imshow(
        input_data[i, :, :, 0],
        cmap="gray"
    )

    plt.axis("off")

plt.suptitle(
    "Preprocessed 16x16 Images"
)

plt.show()

In [ ]:
# ============================================
# CHECK PIXEL LEVELS
# ============================================

unique_values = np.unique(
    output_data
)

print(
    "Unique pixel levels:",
    unique_values
)

In [ ]:
# ============================================
# MASKED CONVOLUTION
# ============================================

class MaskedConv2D(layers.Layer):

    def __init__(
        self,
        mask_type,
        **kwargs
    ):

        super().__init__()

        self.mask_type = mask_type

        self.conv = layers.Conv2D(
            **kwargs
        )

    def build(self, input_shape):

        self.conv.build(
            input_shape
        )

        kernel_shape = (
            self.conv.kernel.shape
        )

        self.mask = np.zeros(
            shape=kernel_shape,
            dtype=np.float32
        )

        # Upper half of kernel
        self.mask[
            :kernel_shape[0] // 2,
            ...
        ] = 1.0

        # Left half of center row
        self.mask[
            kernel_shape[0] // 2,
            :kernel_shape[1] // 2,
            ...
        ] = 1.0

        # Type B includes current pixel
        if self.mask_type == "B":

            self.mask[
                kernel_shape[0] // 2,
                kernel_shape[1] // 2,
                ...
            ] = 1.0

        super().build(input_shape)

    def call(self, inputs):

        self.conv.kernel.assign(
            self.conv.kernel *
            tf.cast(
                self.mask,
                self.conv.kernel.dtype
            )
        )

        return self.conv(inputs)

    def get_config(self):

        config = super().get_config()

        config.update({
            "mask_type": self.mask_type
        })

        return config

In [ ]:
# ============================================
# TEST MASKED CONVOLUTION
# ============================================

test_layer = MaskedConv2D(
    mask_type="A",
    filters=8,
    kernel_size=7,
    padding="same"
)

test_output = test_layer(
    input_data[:2]
)

print(
    "Test output shape:",
    test_output.shape
)

In [ ]:
# ============================================
# RESIDUAL BLOCK
# ============================================

class ResidualBlock(layers.Layer):

    def __init__(
        self,
        filters,
        **kwargs
    ):

        super().__init__(**kwargs)

        self.conv1 = layers.Conv2D(
            filters=filters // 2,
            kernel_size=1,
            activation="relu"
        )

        self.pixel_conv = MaskedConv2D(
            mask_type="B",
            filters=filters // 2,
            kernel_size=3,
            activation="relu",
            padding="same"
        )

        self.conv2 = layers.Conv2D(
            filters=filters,
            kernel_size=1,
            activation="relu"
        )

    def call(self, inputs):

        x = self.conv1(inputs)

        x = self.pixel_conv(x)

        x = self.conv2(x)

        return layers.add(
            [inputs, x]
        )

    def get_config(self):

        return super().get_config()

In [ ]:
# ============================================
# TEST RESIDUAL BLOCK
# ============================================

test_block = ResidualBlock(
    filters=N_FILTERS
)

test_output = test_block(
    input_data[:2]
)

print(
    "Residual block output:",
    test_output.shape
)

In [ ]:
# ============================================
# BUILD PIXELCNN
# ============================================

inputs = layers.Input(
    shape=(
        IMAGE_SIZE,
        IMAGE_SIZE,
        1
    )
)

# First masked convolution
x = MaskedConv2D(
    mask_type="A",
    filters=N_FILTERS,
    kernel_size=7,
    activation="relu",
    padding="same"
)(inputs)

# Residual blocks
for _ in range(RESIDUAL_BLOCKS):

    x = ResidualBlock(
        filters=N_FILTERS
    )(x)

# Additional masked convolution layers
for _ in range(2):

    x = MaskedConv2D(
        mask_type="B",
        filters=N_FILTERS,
        kernel_size=1,
        strides=1,
        activation="relu",
        padding="valid"
    )(x)

# Output layer
out = layers.Conv2D(
    filters=PIXEL_LEVELS,
    kernel_size=1,
    strides=1,
    activation="softmax",
    padding="valid"
)(x)

pixel_cnn = models.Model(
    inputs,
    out
)

pixel_cnn.summary()

In [ ]:
# ============================================
# CHECK MODEL OUTPUT
# ============================================

sample_output = pixel_cnn.predict(
    input_data[:2],
    verbose=0
)

print(
    "Model output shape:",
    sample_output.shape
)

In [ ]:
# ============================================
# COMPILE MODEL
# ============================================

adam = optimizers.Adam(
    learning_rate=0.0005
)

pixel_cnn.compile(
    optimizer=adam,
    loss="sparse_categorical_crossentropy"
)

print("PixelCNN compiled successfully.")

In [ ]:
# ============================================
# CREATE OUTPUT DIRECTORIES
# ============================================

os.makedirs(
    "/content/pixelcnn_output",
    exist_ok=True
)

os.makedirs(
    "/content/pixelcnn_logs",
    exist_ok=True
)

print("Directories created.")

In [ ]:
# ============================================
# IMAGE GENERATOR
# ============================================

class ImageGenerator(
    callbacks.Callback
):

    def __init__(
        self,
        num_img=10,
        save_images=True
    ):

        super().__init__()

        self.num_img = num_img
        self.save_images = save_images

    def sample_from(
        self,
        probs,
        temperature
    ):

        probs = np.asarray(
            probs
        ).astype("float64")

        # Temperature sampling
        probs = np.log(
            probs + 1e-8
        ) / temperature

        exp_probs = np.exp(
            probs - np.max(probs)
        )

        probs = (
            exp_probs /
            np.sum(exp_probs)
        )

        return np.random.choice(
            len(probs),
            p=probs
        )

    def generate(
        self,
        temperature=1.0
    ):

        generated_images = np.zeros(
            shape=(
                self.num_img,
                IMAGE_SIZE,
                IMAGE_SIZE,
                1
            ),
            dtype=np.float32
        )

        for row in range(IMAGE_SIZE):

            for col in range(IMAGE_SIZE):

                # Predict probability distribution
                probs = self.model.predict(
                    generated_images,
                    verbose=0
                )[
                    :,
                    row,
                    col,
                    :
                ]

                # Sample each image
                sampled_pixels = [
                    self.sample_from(
                        p,
                        temperature
                    )
                    for p in probs
                ]

                generated_images[
                    :,
                    row,
                    col,
                    0
                ] = (
                    np.array(
                        sampled_pixels
                    ) /
                    PIXEL_LEVELS
                )

        return generated_images

    def display_images(
        self,
        images,
        epoch=None
    ):

        plt.figure(
            figsize=(12, 5)
        )

        for i in range(
            min(self.num_img, 10)
        ):

            plt.subplot(
                2,
                5,
                i + 1
            )

            plt.imshow(
                images[i, :, :, 0],
                cmap="gray",
                vmin=0,
                vmax=1
            )

            plt.axis("off")

        if epoch is not None:

            plt.suptitle(
                f"Generated Images - Epoch {epoch}"
            )

        plt.tight_layout()

        plt.show()

    def on_epoch_end(
        self,
        epoch,
        logs=None
    ):

        # Generate every 5 epochs
        if (
            (epoch + 1) % 5 == 0
            or epoch == 0
        ):

            print(
                f"\nGenerating images "
                f"after epoch {epoch + 1}..."
            )

            generated_images = self.generate(
                temperature=1.0
            )

            self.display_images(
                generated_images,
                epoch=epoch + 1
            )

In [ ]:
# ============================================
# CALLBACKS
# ============================================

tensorboard_callback = callbacks.TensorBoard(
    log_dir="/content/pixelcnn_logs"
)

img_generator_callback = ImageGenerator(
    num_img=10
)

print("Callbacks created.")

In [ ]:
# ============================================
# TRAIN PIXELCNN
# ============================================

history = pixel_cnn.fit(
    input_data,
    output_data,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=[
        tensorboard_callback,
        img_generator_callback
    ]
)

In [ ]:
# ============================================
# TRAINING LOSS
# ============================================

plt.figure(
    figsize=(10, 5)
)

plt.plot(
    history.history["loss"],
    marker="o"
)

plt.title(
    "PixelCNN Training Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.grid(True)

plt.show()

In [ ]:
# ============================================
# GENERATE NEW IMAGES
# ============================================

generated_images = (
    img_generator_callback.generate(
        temperature=1.0
    )
)

print(
    "Generated images shape:",
    generated_images.shape
)

In [ ]:
# ============================================
# DISPLAY GENERATED IMAGES
# ============================================

img_generator_callback.display_images(
    generated_images
)

In [ ]:
# ============================================
# TEMPERATURE COMPARISON
# ============================================

temperatures = [
    0.5,
    1.0,
    1.5
]

for temperature in temperatures:

    print(
        f"\nTemperature = {temperature}"
    )

    images = (
        img_generator_callback.generate(
            temperature=temperature
        )
    )

    img_generator_callback.display_images(
        images
    )

In [ ]:
# ============================================
# SAVE GENERATED IMAGES
# ============================================

import matplotlib.pyplot as plt

for i in range(
    len(generated_images)
):

    plt.imsave(
        f"/content/pixelcnn_output/"
        f"generated_{i}.png",
        generated_images[i, :, :, 0],
        cmap="gray",
        vmin=0,
        vmax=1
    )

print(
    "Images saved to:"
)

print(
    "/content/pixelcnn_output/"
)

In [ ]:
# ============================================
# SAVE PIXELCNN MODEL
# ============================================

MODEL_PATH = (
    "/content/pixelcnn_model.keras"
)

pixel_cnn.save(
    MODEL_PATH
)

print(
    "Model saved successfully!"
)

print(
    "Path:",
    MODEL_PATH
)